# 08 — Evaluation

Benchmark the fine-tuned model against GPT-4o zero-shot and a plain-text baseline.
Also evaluate catastrophic forgetting — how much general capability the model retained.

**Metrics**:
- **Schema validity rate** — % of outputs that parse as valid JSON-LD
- **Property coverage** — avg properties per example vs gold standard
- **Type accuracy** — correct schema @type for the page
- **BLEU/ROUGE** — output similarity to gold (crude but fast)
- **Property F1** — precision/recall on extracted property names
- **Forgetting delta** — performance on general QA after fine-tuning vs before

**Connection to dissertation**: The forgetting delta section directly measures
catastrophic forgetting in the fine-tuned 7B model — relevant to the EWC/LoRA chapter.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from collections import defaultdict
import numpy as np
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

from src.schema_validator import validate_jsonld
print('Imports OK')

## Load Held-Out Test Set

In [ ]:
# Load eval set from notebook 05
eval_path = Path('../data/processed/eval.jsonl')

eval_examples = []
with open(eval_path) as f:
    for line in f:
        eval_examples.append(json.loads(line))

# Sample 200 for fast evaluation
import random
random.seed(42)
test_set = random.sample(eval_examples, min(200, len(eval_examples)))
print(f'Test set size: {len(test_set)}')

## Helper: Evaluation Metrics

In [ ]:
def extract_properties(jsonld_str: str) -> set:
    """Extract property names from a JSON-LD string."""
    try:
        obj = json.loads(jsonld_str)
        exclude = {'@context', '@type', '@id', '@graph'}
        return {k for k in obj.keys() if k not in exclude}
    except:
        return set()

def property_f1(pred_str: str, gold_str: str) -> dict:
    """Compute precision, recall, F1 on property names."""
    pred_props = extract_properties(pred_str)
    gold_props = extract_properties(gold_str)
    
    if not gold_props:
        return {'precision': 0, 'recall': 0, 'f1': 0}
    
    tp = len(pred_props & gold_props)
    precision = tp / len(pred_props) if pred_props else 0
    recall = tp / len(gold_props)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3)}

def type_match(pred_str: str, gold_str: str) -> bool:
    """Check if predicted @type matches gold @type."""
    try:
        pred = json.loads(pred_str)
        gold = json.loads(gold_str)
        pred_type = pred.get('@type')
        gold_type = gold.get('@type')
        if pred_type and gold_type:
            pred_types = [pred_type] if isinstance(pred_type, str) else pred_type
            gold_types = [gold_type] if isinstance(gold_type, str) else gold_type
            return bool(set(pred_types) & set(gold_types))
    except:
        pass
    return False

def evaluate_predictions(predictions: list[dict]) -> dict:
    """
    Evaluate a list of {pred, gold} dicts.
    Returns aggregate metrics.
    """
    validity_scores = []
    quality_scores = []
    type_matches = []
    property_f1s = []
    
    for item in predictions:
        pred = item.get('pred', '')
        gold = item.get('gold', '')
        
        if not pred:
            validity_scores.append(0)
            quality_scores.append(0)
            type_matches.append(False)
            property_f1s.append(0)
            continue
        
        val = validate_jsonld(pred)
        validity_scores.append(1 if val['valid'] else 0)
        quality_scores.append(val['quality_score'])
        type_matches.append(type_match(pred, gold))
        pf1 = property_f1(pred, gold)
        property_f1s.append(pf1['f1'])
    
    return {
        'n': len(predictions),
        'validity_rate': round(np.mean(validity_scores), 3),
        'avg_quality_score': round(np.mean(quality_scores), 3),
        'type_accuracy': round(np.mean(type_matches), 3),
        'avg_property_f1': round(np.mean(property_f1s), 3),
    }

print('Evaluation helpers defined')

## Evaluate Fine-Tuned Model (on RunPod)

In [ ]:
# This section runs on the RunPod pod with the fine-tuned model loaded
# On local machine: use the RunPod serverless endpoint instead

RUNPOD_ENDPOINT_ID = os.getenv('RUNPOD_ENDPOINT_ID')

def get_model_prediction(example: dict, mode: str = 'endpoint') -> str:
    """
    Get prediction from fine-tuned model.
    mode='endpoint': use RunPod serverless endpoint
    mode='local': use local vLLM (on GPU pod)
    """
    messages = example.get('messages', [])
    user_msg = next((m for m in messages if m['role'] == 'user'), None)
    if not user_msg:
        return ''
    
    html = ''
    for content_item in user_msg.get('content', []):
        if isinstance(content_item, dict) and content_item.get('type') == 'text':
            text = content_item['text']
            if 'HTML:' in text:
                html = text.split('HTML:\n', 1)[-1]
    
    if mode == 'endpoint' and RUNPOD_ENDPOINT_ID:
        from src.runpod_utils import submit_serverless_job, init_runpod
        init_runpod()
        result = submit_serverless_job(RUNPOD_ENDPOINT_ID, {'html': html})
        return result.get('schema_jsonld', '') if result else ''
    
    # Local mode — assumes model is loaded
    # return engine.generate(html)
    return ''

print(f'RunPod endpoint: {RUNPOD_ENDPOINT_ID or "Not configured"}')
print('Set RUNPOD_ENDPOINT_ID in .env and run notebook 09 to deploy the endpoint first')

## GPT-4o Baseline (Zero-Shot)

In [ ]:
# Compare against GPT-4o zero-shot as an upper baseline
# Cost: ~$0.05-0.25/page, so cap at 50 examples for the comparison

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

def gpt4o_baseline(html: str) -> str:
    """Generate schema with GPT-4o zero-shot for comparison."""
    if not OPENAI_API_KEY:
        return ''
    
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    prompt = (
        'Generate optimal schema.org JSON-LD for this web page. '
        'Output ONLY valid JSON-LD, no markdown.\n\nHTML:\n' + html[:8000]
    )
    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=1024,
        temperature=0.1,
    )
    return resp.choices[0].message.content.strip()

print('GPT-4o baseline function defined')
print(f'OpenAI API key present: {bool(OPENAI_API_KEY)}')

# Run GPT-4o on 50 examples
RUN_GPT4O = False  # Set True to run (~$5-10)
gpt4o_preds = []

if RUN_GPT4O and OPENAI_API_KEY:
    small_test = test_set[:50]
    for ex in small_test:
        messages = ex.get('messages', [])
        user_msg = next((m for m in messages if m['role'] == 'user'), None)
        html = ''
        if user_msg:
            for c in user_msg.get('content', []):
                if isinstance(c, dict) and c.get('type') == 'text' and 'HTML:' in c['text']:
                    html = c['text'].split('HTML:\n', 1)[-1]
        
        gold = next((m['content'] for m in messages if m['role'] == 'assistant'), '')
        pred = gpt4o_baseline(html)
        gpt4o_preds.append({'pred': pred, 'gold': gold})
    
    gpt4o_metrics = evaluate_predictions(gpt4o_preds)
    print('GPT-4o zero-shot metrics:', gpt4o_metrics)

## Results Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Placeholder results — populate after running evaluations
results = {
    'GPT-4o zero-shot': {'validity_rate': 0.0, 'avg_quality_score': 0.0, 'type_accuracy': 0.0, 'avg_property_f1': 0.0},
    'Fine-tuned (ours)': {'validity_rate': 0.0, 'avg_quality_score': 0.0, 'type_accuracy': 0.0, 'avg_property_f1': 0.0},
}

# Uncomment and fill in when results are available:
# results['GPT-4o zero-shot'] = gpt4o_metrics
# results['Fine-tuned (ours)'] = finetuned_metrics

df_results = pd.DataFrame(results).T
print('Evaluation Results:')
print(df_results.to_string())

# Bar chart
metrics = ['validity_rate', 'avg_quality_score', 'type_accuracy', 'avg_property_f1']
df_results[metrics].plot(kind='bar', figsize=(12, 5), ylim=(0, 1))
plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Catastrophic Forgetting Assessment

Does fine-tuning on schema.org data hurt general web understanding?

In [ ]:
# Simple forgetting test: ask model general web-related questions
# Compare responses before and after fine-tuning on standard benchmarks

GENERAL_QA_PROMPTS = [
    "What is the purpose of the <meta> tag in HTML?",
    "Explain the difference between GET and POST HTTP methods.",
    "What does SEO stand for and why does it matter?",
    "What is the difference between a URL and a URI?",
    "Describe what JSON-LD is and why it's preferred over Microdata.",
]

# For a rigorous forgetting analysis, run these on:
# 1. Base Qwen2.5-VL-7B (before fine-tuning)
# 2. Fine-tuned model (after)
# Compare response quality with a judge model (GPT-4o or Claude)

print('Catastrophic forgetting test prompts defined.')
print('Run against base model and fine-tuned model; compare with judge LLM.')
print('This forms the basis of the dissertation case study.')
print('\nNext step: 09_irish_web_inference.ipynb')